# Transformers


In [ ]:
#| default_exp transformers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Optional
import torch
from torch import nn
from torch import Tensor
import warnings, copy, numpy as np
from physiojepa.layers import *

## Time Series Transformer Encoder

In [ ]:
#| export
class TSTEncoderLayer(nn.Module):
    def __init__(self, 
                 d_model, # dimension of patch embeddings
                 n_heads, # number of attention heads per layer
                 d_ff=256, # dimension of feedforward layer in each transformer layer
                 attn_dropout=0, 
                 dropout=0., 
                 bias=True, 
                 activation="gelu", 
                 pre_norm=False
                ):
        super().__init__()
        
        assert not d_model%n_heads, f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"

        # Multi-Head attention
        self.self_attn = MultiheadFlashAttention(d_model, n_heads, attn_dropout=attn_dropout, proj_dropout=dropout)

        # Add & Norm
        self.dropout_attn = nn.Dropout(dropout) 
        self.norm_attn = nn.LayerNorm(d_model)

        # Position-wise Feed-Forward
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff, bias=bias), 
                                get_activation_fn(activation), # note do not put functions in sequential, it makes things non-deterministic
                                nn.Dropout(dropout),
                                nn.Linear(d_ff, d_model, bias=bias))

        # Add & Norm
        self.dropout_ffn = nn.Dropout(dropout)
        self.norm_ffn = nn.LayerNorm(d_model)

        self.pre_norm = pre_norm


    def forward(self, src:Tensor, channel_mask:Optional[Tensor]=None):
        """
        src: tensor [bs x q_len x d_model]
        channel_mask: tensor [bs x n_channels]
        """
        # Multi-Head attention sublayer
        if self.pre_norm:
            src = self.norm_attn(src)
        ## Multi-Head attention
        src2 = self.self_attn(src, channel_mask=channel_mask)
        
        ## Add & Norm
        src = src + self.dropout_attn(src2) # Add: residual connection with residual dropout
        if not self.pre_norm:
            src = self.norm_attn(src)
        # Feed-forward sublayer
        if self.pre_norm:
            src = self.norm_ffn(src)
        ## Position-wise Feed-Forward

        src2 = self.ff(src)

        ## Add & Norm
        src = src + self.dropout_ffn(src2) # Add: residual connection with residual dropout
        if not self.pre_norm:
            src = self.norm_ffn(src)

        return src

## PhysioJEPA

In [ ]:
#| export
class PatchTSJEPAPredictor(nn.Module):
    """Predictor network for PatchTSJEPA encoder for time series adaptation"""
    def __init__(
        self,
        c_in,                # number of input channels
        num_patches,         # number of patches from encoder
        d_model=512,        # encoder embedding dimension
        predictor_dim=384,  # predictor embedding dimension (typically smaller)
        n_heads=4,
        n_layers=2,
        d_ff=1024,
        pos_encoding_type='learned', # 'learned' or 'tAPE'
        dropout=0.1,
        attn_dropout=0.0,
        act="gelu",
        pre_norm=False,
    ):
        super().__init__()
        
        self.c_in = c_in
        self.num_patches = num_patches
        self.predictor_dim = predictor_dim
        self.d_model = d_model
        
        # Project from encoder dimension to predictor dimension
        self.predictor_embed = nn.Linear(d_model, predictor_dim, bias=True)
        
        # Learnable mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        torch.nn.init.normal_(self.mask_token, std=.02)
        
        # Positional encoding for predictor
        if pos_encoding_type.lower() == 'tape':
            W_pos = torch.zeros(1, num_patches, predictor_dim)  # positional encoding
            position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, predictor_dim, 2).float() * (-np.log(10000.0) / predictor_dim))

            W_pos[..., 0::2] = torch.sin((position * div_term)*(predictor_dim/num_patches)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
            W_pos[..., 1::2] = torch.cos((position * div_term)*(predictor_dim/num_patches))
            self.register_buffer('W_pos', W_pos)  # this stores the variable in the state_dict (used for non-trainable variables)
        else:
            self.W_pos =  nn.Parameter(torch.empty((1, num_patches, predictor_dim)))
            nn.init.uniform_(self.W_pos, -0.02, 0.02)
        self.pe_scale_factor = nn.Parameter(torch.ones(1))
        # Predictor transformer layers (using same TSTEncoderLayer as encoder)
        self.predictor_layers = nn.ModuleList([
            TSTEncoderLayer(
                d_model=predictor_dim,
                n_heads=n_heads,
                d_ff=d_ff,
                attn_dropout=attn_dropout,
                dropout=dropout,
                activation=act,
                pre_norm=pre_norm
            ) for _ in range(n_layers)
        ])
        
        # Final projection back to encoder dimension
        self.predictor_norm = nn.LayerNorm(predictor_dim)
        self.predictor_proj = nn.Linear(predictor_dim, d_model, bias=True)
        
    def forward(self, x, masks_x=None, masks=None):
        """
        Args:
            x: Encoded patches from encoder [B, C, d_model, n_patches]
            masks_x: Context masks - which patches are visible to predictor
            masks: Target masks - which patches to predict
        Returns:
            Predictions for masked patches [B, C, d_model, n_masked_patches]
        """
        B = x.size(0)
        
        # Reshape and project to predictor dimension
        x = x.permute(0, 3, 1, 2) # [B, n_patches, C, d_model]
        x = self.predictor_embed(x)  # [B, n_patches, C, predictor_dim] 
        x = x.permute(0,2,1,3)
        if not x.is_nested:
            x = torch.reshape(x, (B * self.c_in, -1, self.predictor_dim)) # u: [bs * nvars x num_patch x d_model]
        else:
            x_list = list(x.unbind())
            reshaped_list = []
            for x_i in x_list:
                # x has shape [c_in, var_len, d_model]
                # Split into 7 separate tensors
                channel_tensors = list(x_i.unbind(0))
                reshaped_list.extend(channel_tensors)
            x = torch.nested.as_nested_tensor(reshaped_list, layout=torch.jagged)
        # Handle visible patches (context)
        if masks_x is not None:
            # Expand mask for all channels
            # masks_x is [B, C, n_patches]
            pos_embed = self.W_pos.repeat(B * self.c_in, 1, 1)  # [B*C, n_patches, predictor_dim]
            
            context_mask = masks_x
            context_mask = context_mask.reshape(B * self.c_in, -1, 1)  # [B*C, n_patches, 1]
            pos_embed = pos_embed.masked_fill(~context_mask, 0.0)  # Zero out non-context positions
            if x.is_nested:
                x_list = list(x.unbind())
                for i in range(len(x_list)):
                    seq_len = x_list[i].size(0)
                    x_list[i] = x_list[i] + pos_embed[i,:seq_len,:] * self.pe_scale_factor # [B*C, n_patches, predictor_dim]
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            else:
                x = x + pos_embed * self.pe_scale_factor # [B*C, n_patches, predictor_dim]

        # Add mask tokens for prediction
        if masks is not None:
            # Get positions that need to be predicted (any channel masked)
            # masks  is [B, C, n_patches]
            target_mask = masks
            target_mask = target_mask.reshape(B * self.c_in, -1, 1)  # [B*C x n_patches x 1]
            target_mask = target_mask.expand(-1, -1, self.predictor_dim)  # [B*C, n_patches, predictor_dim]

            # Create positional embeddings for masked positions
            pos_embs = self.W_pos.repeat(B*self.c_in, 1, 1) # [B*C, num_patches, predictor_dim]

            mask_tokens = self.mask_token.expand_as(pos_embs)
            mask_tokens = mask_tokens + pos_embs  # Add positional embeddings
            # Zero out positions we don't want to predict
            # in the target mask, True means what we want to predict so we do not need to invert it.
            if x.is_nested:
                x_list = list(x.unbind())
                for i in range(len(x_list)):
                    seq_len = x_list[i].size(0)
                    x_list[i] = torch.where(target_mask[i, :seq_len, :], mask_tokens[i, :seq_len, :], x_list[i])
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            else:
                x = torch.where(target_mask, mask_tokens, x)

        # Forward through transformer layers
        for layer in self.predictor_layers:
            x = layer(x)  # No need for padding mask since we handle masking explicitly
        # Final norm and projection
        x = self.predictor_norm(x)

        if x.is_nested:
            x_list = list(x.unbind())
            restored_list = []
            for i in range(0, len(x_list), self.c_in):
                group = x_list[i:i+self.c_in]
                stacked = torch.stack(group, dim=0)
                stacked = stacked.reshape(-1, self.predictor_dim) # z: [nvars * num_patch x d_model]
                restored_list.append(stacked)
            x = torch.nested.as_nested_tensor(restored_list, layout=torch.jagged)
        else:
            x = x.reshape(B, self.c_in, -1, x.shape[-1])  # [B, C, n_patches, pred dim]
        x = self.predictor_proj(x)  # Project back to encoder dimension
        if x.is_nested:
            x_list = list(x.unbind())
            for i in range(len(x_list)):
                x_list[i] = x_list[i].reshape(self.c_in, -1, self.d_model) # reshape to [bs x nvars x n patch x predict dim]
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        # Reshape back
        # Only return predictions for masked tokens
        if masks is not None:
            target_mask = masks
            if x.is_nested:
                x_list = list(x.unbind())
                for i in range(len(x_list)):
                    seq_len = x_list[i].size(1)
                    x_list[i] = x_list[i][target_mask[i,:, :seq_len]]
                    x_list[i] = x_list[i].reshape(self.c_in, -1, self.d_model)
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            else:
                x = x[target_mask]
                x = x.reshape(B, self.c_in, -1, x.shape[-1])
            x = x.transpose(-1,-2)
        return x

In [ ]:
#| export
class PatchTSJEPAEncoder(nn.Module):
     def __init__(self,
                  c_in:int, # the number of input channels
                  win_length, # the length of the patch of time/interval or short time ft windown length (when time_domain=False)
                  hop_length, # the length of the distance between each patch/fft
                  max_seq_len, # maximum sequence len
                  pos_encoding_type='learned', # 'learned' or 'tAPE'
                  patch_encoder_type='linear', # 'linear' or 'conv'
                  use_revin=True, # if time_domain is true, whether or not to instance normalize time data
                  affine=True, # if time_domain is true, whether or not to learn revin normalization parameters 
                  n_layers:int=4, # the number of transformer encoder layers to use
                  d_model=512, # the dimension of the input to the transofmrer encoder
                  n_heads=8, # the number of heads in each layer
                  shared_embedding=False, # indicator for whether or not each channel should be projected with its own set of linear weights to the encoder dimension
                  d_ff:int=2048, # the feedforward layer size in the transformer
                  attn_dropout:float=0., # dropout in attention
                  dropout:float=0.1, # dropout for linear layers
                  act:str="gelu", # activation function
                  pre_norm:bool=False, # indicator to pre batch or layer norm 
                  ):
          super().__init__()
          self.c_in = c_in # original c_in without convolution
          self.shared_embedding = shared_embedding
          self.d_model = d_model # original d_model
          self.use_revin = use_revin
          self.affine = affine
          self.hop_length = hop_length
          self.max_seq_len = max_seq_len
          # Instance Normalization (full sequence)
          if self.use_revin:
               self.revin = RevIN(num_features=self.c_in, affine=self.affine, dim_to_reduce=-1)
          self.num_patch = int((max(max_seq_len, win_length)-win_length) // hop_length + 1)
          if ((max_seq_len-win_length) % hop_length != 0):
               # add one for padding if above is true, see create_patch fxn for more details
               self.num_patch += 1
          self.patch_len = win_length
          self.patch_layer = Patch(patch_len=win_length, stride=hop_length)
          
          # Patch Embedding
          if patch_encoder_type == 'linear':
               self.patch_encoder = PatchEncoder(c_in=self.c_in, patch_len=self.patch_len, d_model = self.d_model, shared_embedding=shared_embedding)
          elif patch_encoder_type == 'conv':
               if self.shared_embedding:
                    warnings.warn("Shared embedding is not yet implemented for conv patch encoder, setting shared_embedding=False")
                    self.shared_embedding = False
               self.patch_encoder = PatchEncoderConv(c_in=self.c_in, patch_len=self.patch_len, kernel_size=20, d_model = self.d_model, shared_embedding=shared_embedding)
          # Positional Encoding
          if pos_encoding_type.lower() == 'tape':
               self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patch)
          else:
               self.pe = PositionalEncoding(num_patch=self.num_patch, d_model=self.d_model)
          # residual dropout
          self.dropout = nn.Dropout(dropout)
          # time series transformer layers/Encoder
          self.layers = nn.ModuleList([TSTEncoderLayer(d_model=self.d_model, n_heads=n_heads, d_ff=d_ff,
                                                       attn_dropout=attn_dropout, dropout=dropout,activation=act,
                                                       pre_norm=pre_norm) for _ in range(n_layers)])

     def forward(self, z, mask=None, channel_mask=None):
          """
          input from ds is [bs x n_vars x max_seq_len]
          z: tensor [bs x nvars x d_model x num_patch]
          mask: tensor [bs x n_vars x num_patch]
          """
          bs = z.size(0)
          # REVIN
          if self.use_revin:
               z = self.revin(z, mode=True) # z: [bs x n_vars x max_seq_len] dont passs sequence pad mask to revin if dim=(1,) - it doesnt matter
          z = self.patch_layer(z, constant_pad=True, constant_pad_value=0) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values
          # MASKING
          if mask is not None:
               # Expand mask for all channels
               mask = mask.transpose(-1,-2) # [bs x num_patch x n_vars]
               mask = mask.unsqueeze(-1) # [bs x num_patch x n_vars x 1]
               #mask = mask.expand(-1, -1, -1, self.patch_len)  # [B, num_patch, n_vars, patch_len]
               # Zero out non-masked patches (this is the context)
               # invert the mask, if mask == 0.5, then 50% of patches are set to True
               if z.is_nested:
                    z_list = list(z.unbind())
                    for i in range(len(z_list)):
                         seq_len = z_list[i].size(0)
                         z_list[i] = z_list[i].masked_fill(~mask[i,:seq_len, :, :], 0.0)
                    z = torch.nested.as_nested_tensor(z_list, layout=torch.jagged)
               else:
                    z = z.masked_fill(~mask, 0.0)  # Now zeros out everything NOT in context

          # EMBEDDING
          z = self.patch_encoder(z) # z: [bs x num_patch x nvars x d_model]
          z = z.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          # positional encoding
          if not z.is_nested:
               z = torch.reshape(z, (bs * self.c_in, self.num_patch, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          else:
               z_list = list(z.unbind())
               reshaped_list = []
               for x in z_list:
                    # x has shape [c_in, var_len, d_model]
                    # Split into 7 separate tensors
                    channel_tensors = list(x.unbind(0))
                    reshaped_list.extend(channel_tensors)
               z = torch.nested.as_nested_tensor(reshaped_list, layout=torch.jagged)
          z = self.pe(z) # z: [bs * nvars x num_patch x d_model]
          # residual dropout
          z = self.dropout(z) # z: [bs * nvars x num_patch x d_model] 
          # encoder layers
          for mod in self.layers: 
               z = mod(z, channel_mask=channel_mask) # z: [bs * n_vars x num_patch x d_model]
          if z.is_nested:
               z_list = list(z.unbind())
               restored_list = []
               for i in range(0, len(z_list), self.c_in):
                    group = z_list[i:i+self.c_in]
                    stacked = torch.stack(group, dim=0).transpose(0,1) # 1st dim is the jagged dimension
                    restored_list.append(stacked)
               z = torch.nested.as_nested_tensor(restored_list, layout=torch.jagged)
               z = z.transpose(1,2)
          else:
               z = torch.reshape(z, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z = z.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch] 
          return z

In [ ]:
#| export
class PatchTSJEPA(nn.Module):
     def __init__(self, 
                  encoder_kwargs: dict, 
                  predictor_kwargs: dict, 
                  pretrain=True, 
                  target_mask_range=(0.1,0.5), # the target can be up to 50% of the original x 
                  context_mask_range=(0.2,0.8), # the context can be up to 80% of masked out target (1-target_mask_ratio)
                  ):
          super().__init__()
          self.context_encoder = PatchTSJEPAEncoder(**encoder_kwargs)
          self.target_encoder = copy.deepcopy(self.context_encoder).requires_grad_(False) # no grad, ema weight updates
          self.predictor = PatchTSJEPAPredictor(**predictor_kwargs)
          self.pretrain = pretrain
          self.target_mask_range = target_mask_range
          self.context_mask_range = context_mask_range
          self.num_patch = predictor_kwargs['num_patches']
          self.c_in = encoder_kwargs['c_in']
          self.win_length = encoder_kwargs['win_length']
          self.hop_length = encoder_kwargs['hop_length']
          self.max_seq_len = encoder_kwargs['max_seq_len']

     def create_masks(self, x):
          """Create context and target masks for I-JEPA training
          Args:
               batch_size: int
          Returns:
               context_mask: bs x n_vars x num_patch boolean mask where True indicates patches visible to predictor
               target_mask: bs x n_vars x num_patch boolean mask where True indicates patches to predict
          """
          bs = x.size(0)

          nested_patches = []
          if x.is_nested:
               x_list = list(x.unbind())
               target_mask_list = []
               for x_i in x_list:
                    n_patches = int((x_i.size(-1) - self.win_length) // self.hop_length + 1)
                    if ((x_i.size(-1)-self.win_length) % self.hop_length != 0):
                         n_patches += 1
                    nested_patches.append(n_patches)
                    target_mask_list.append(torch.zeros((n_patches, self.c_in), dtype=torch.bool))
               target_mask = torch.nested.as_nested_tensor(target_mask_list, layout=torch.jagged)
               target_mask = target_mask.transpose(1,2) # transpose so that the jagged dimension is the first dimension
          else:
               target_mask = torch.zeros((bs, self.c_in, self.num_patch), dtype=torch.bool) # [bs x n_vars x num_patch x patch_len]
          context_mask = torch.zeros_like(target_mask) # [bs x n_vars x num_patch x patch_len]
          
          # target and context mask ratios, should be the same for each batch item
          target_ratio = self.target_mask_range[0] + torch.rand(1).item() * (self.target_mask_range[1] - self.target_mask_range[0])
          context_ratio = self.context_mask_range[0] + torch.rand(1).item() * (self.context_mask_range[1] - self.context_mask_range[0])
          if x.is_nested:
               x_list = list(x.unbind())
               target_list = list(target_mask.unbind())
               context_list = list(context_mask.unbind())
               for i,x_i in enumerate(x_list):
                    perm = torch.randperm(nested_patches[i])
                    num_target = int(nested_patches[i] * target_ratio)
                    remaining = nested_patches[i] - num_target
                    num_context = int(remaining * context_ratio)
                    target_indices = perm[:num_target]
                    target_list[i][:, target_indices] = True
                    context_indices = perm[num_target:num_target + num_context]
                    context_list[i][:, context_indices] = True
                    assert torch.all((target_list[i] == True) & (context_list[i] == True)) == False, "target and context masks should not overlap"
               ## finally convert them to padded tensors
               context_mask = context_mask.to_padded_tensor(padding=0, output_size=(bs,self.c_in,self.num_patch))
               target_mask = target_mask.to_padded_tensor(padding=0, output_size=(bs,self.c_in,self.num_patch))
          else:
               perm = torch.randperm(self.num_patch)
               num_target = int(self.num_patch * target_ratio)
               remaining = self.num_patch - num_target
               num_context = int(remaining * context_ratio)
                    
               target_indices = perm[:num_target]
               context_indices = perm[num_target:num_target + num_context]
               
               for i in range(bs):
                    target_mask[i, :, target_indices] = True
                    context_mask[i, :, context_indices] = True
               
               assert torch.all((target_mask == True) & (context_mask == True)) == False, "target and context masks should not overlap"
          # here, True means the values that are not masked out!!! need to be careful with masking functions in Torch. 
          ## torch.where does not need an inversion, torch.masked_fill does need an inversion
          return context_mask, target_mask
     
     def _pretraining_forward(self, x):
          """Forward pass during pretraining"""
          # Create masks and prepare inputs
          context_mask, target_mask = self.create_masks(x)
          context_mask = context_mask.to(x.device)
          target_mask = target_mask.to(x.device)
          
          
          # Get encoded representations and apply masks
          z_context = self.context_encoder(x, mask=context_mask)
          
          with torch.no_grad():
               self.target_encoder.eval()
               z_target = self.target_encoder(x, mask=None) # # z: [bs x nvars x d_model x num_patch] 
          
          # Process target mask and select masked patches
          bs = z_target.size(0)
          
          z_target = z_target.transpose(-1,-2) # bs x nvars x num_patch x d_model
          if z_target.is_nested:
               z_target_list = list(z_target.unbind())
               for i in range(len(z_target_list)):
                    seq_len = z_target_list[i].size(1)
                    z_target_list[i] = z_target_list[i][target_mask[i,:,:seq_len]]
                    z_target_list[i] = z_target_list[i].reshape(self.c_in, -1, z_target_list[i].size(1))
               z_target = torch.nested.as_nested_tensor(z_target_list, layout=torch.jagged)
          else:
               z_target = z_target[target_mask]
               z_target = z_target.reshape(bs, self.c_in, -1, z_target.size(1))
          z_target = z_target.transpose(-1,-2) # bs x nvars x d_model x num_patch
          # Get predictions
          pred = self.predictor(z_context, masks_x=context_mask, masks=target_mask)
               
          return pred, z_target, z_context, context_mask, target_mask
     
     def _inference_forward(self, x, channel_mask=None):
        """Forward pass during inference"""
        self.target_encoder.eval()
        z = self.target_encoder(x, channel_mask=channel_mask)
        return z

     def forward(self, x, channel_mask=None):
        """Main forward pass"""
        if self.pretrain or self.training:
            return self._pretraining_forward(x)
        else:
            return self._inference_forward(x, channel_mask=channel_mask)

In [ ]:
#| notest
d_model = 1024
encoder_kwargs = {'c_in': 7, 'win_length': 500, 'hop_length': 500, 'max_seq_len': 1*3600*100, 'use_revin': True, 'affine': True, 'n_layers': 4, 'd_model': d_model, 'n_heads': 8, 'shared_embedding': False, 'd_ff': 2048, 'attn_dropout': 0., 'dropout': 0.1, 'act': "gelu", 'pre_norm': False}
predictor_kwargs = {'c_in': 7, 'num_patches': 720, 'd_model': d_model, 'predictor_dim': 256, 'n_heads': 4, 'n_layers': 2, 'd_ff': 1024, 'dropout': 0.1, 'attn_dropout': 0., 'act': "gelu", 'pre_norm': False}
model = PatchTSJEPA(encoder_kwargs=encoder_kwargs, predictor_kwargs=predictor_kwargs, pretrain=True, target_mask_range=(0.1,0.5), context_mask_range=(0.2,0.8))

# x = torch.randn(2, 7, 1*3600*100)

# pred, z_target, z_context, context_mask, target_mask = model(x)

# pred.shape, z_target.shape, z_context.shape, context_mask.shape, target_mask.shape


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()